# Model Evaluation

Plan: Train Model on 2020-2023 data, predict on 2024

### Key Metrics
- RMSE
    - Overall
    - By Position
    - By Week/As Season Progresses
- Binary Accuracy
    - Predict on Similar Players and 
    - Gives a Realistic Evaluation of How It Performs In Deployment

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


ModuleNotFoundError: No module named 'pandas'

# Create Train/Test Split 

- Train on 2020-2023 dat
- Test on 2024

In [25]:
training = pd.read_csv('~/Desktop/projects/Fantasy-Football-Predictor-2025/data/training.csv')
training = training.drop(['Unnamed: 0', 'Unnamed: 0_last1.1'], axis = 1)
training = pd.get_dummies(training, columns= ['position'])
# create target column
weekly = pd.read_csv('~/Desktop/projects/Fantasy-Football-Predictor-2025/data/weekly.csv')
training['fantasy_points_ppr'] = weekly['fantasy_points_ppr']


In [26]:
# create training and testing
train_data = training[training['season'] < 2024]
# train on 2020-2023, test on 2024
test_data = training[training['season'] == 2024]

train_data.set_index(['player_name', 'team', 'season', 'week'], inplace = True)
test_data.set_index(['player_name', 'team', 'season', 'week'], inplace = True)

In [27]:
train_data

,,,,passing_yards_avg,passing_tds_avg,interceptions_avg,passing_epa_avg,carries_avg,rushing_yards_avg,rushing_tds_avg,fumbles_avg,fumbles_lost_avg,rushing_epa_avg,...,is_home,spread_line,total_line,implied_team_total,position_FB,position_QB,position_RB,position_TE,position_WR,fantasy_points_ppr
player_name,team,season,week,,,,,,,,,,,,,,,,,,,,,
Chase Edmonds,ARI,2020,1,0.000000,0.000000,0.000000,0.000000,6.000000,30.300000,0.400000,0.000000,0.000000,0.265610,...,0,7.0,48.5,20.75,0,0,1,0,0,25.64
Christian Kirk,ARI,2020,1,0.000000,0.000000,0.000000,0.000000,0.769231,7.153846,0.000000,0.000000,0.000000,0.492844,...,0,7.0,48.5,20.75,0,0,0,0,1,24.66
Dan Arnold,ARI,2020,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,7.0,48.5,20.75,0,0,0,1,0,20.14
DeAndre Hopkins,ARI,2020,1,0.352941,0.058824,0.058824,-2.789241,0.117647,1.058824,0.000000,0.117647,0.058824,0.988049,...,0,7.0,48.5,20.75,0,0,0,0,1,3.70
Kenyan Drake,ARI,2020,1,0.000000,0.000000,0.000000,0.000000,12.000000,56.071429,0.571429,0.142857,0.071429,-0.044920,...,0,7.0,48.5,20.75,0,0,1,0,0,23.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Jahan Dotson,WAS,2023,17,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,-14.0,49.0,17.50,0,0,0,0,1,9.90
John Bates,WAS,2023,17,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,-14.0,49.0,17.50,0,0,0,1,0,11.20
Logan Thomas,WAS,2023,17,0.000000,0.000000,0.000000,0.000000,0.071429,0.142857,0.000000,0.142857,0.142857,0.510728,...,1,-14.0,49.0,17.50,0,0,0,1,0,12.60


In [28]:
test_data

,,,,passing_yards_avg,passing_tds_avg,interceptions_avg,passing_epa_avg,carries_avg,rushing_yards_avg,rushing_tds_avg,fumbles_avg,fumbles_lost_avg,rushing_epa_avg,...,is_home,spread_line,total_line,implied_team_total,position_FB,position_QB,position_RB,position_TE,position_WR,fantasy_points_ppr
player_name,team,season,week,,,,,,,,,,,,,,,,,,,,,
DeeJay Dallas,ARI,2024,1,0.0,0.0,0.0,-0.517852,1.000000,3.600000,0.000000,0.000000,0.000000,0.022323,...,0,6.5,46.0,19.75,0,0,1,0,0,42.62
Elijah Higgins,ARI,2024,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,6.5,46.0,19.75,0,0,0,1,0,39.42
Emari Demercado,ARI,2024,1,0.0,0.0,0.0,0.000000,5.272727,25.818182,0.181818,0.000000,0.000000,-0.529673,...,0,6.5,46.0,19.75,0,0,1,0,0,13.06
Greg Dortch,ARI,2024,1,0.0,0.0,0.0,0.000000,0.111111,0.555556,0.000000,0.000000,0.000000,-0.083164,...,0,6.5,46.0,19.75,0,0,0,0,1,14.76
James Conner,ARI,2024,1,0.0,0.0,0.0,0.000000,16.000000,80.000000,0.538462,0.000000,0.000000,1.072543,...,0,6.5,46.0,19.75,0,0,1,0,0,16.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Jeremy McNichols,WAS,2024,17,0.0,0.0,0.0,0.000000,4.153846,19.692308,0.307692,0.000000,0.000000,0.630134,...,1,3.5,46.5,25.00,0,0,1,0,0,9.20
John Bates,WAS,2024,17,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.125000,0.125000,0.000000,...,1,3.5,46.5,25.00,0,0,0,1,0,6.70
Olamide Zaccheaus,WAS,2024,17,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,3.5,46.5,25.00,0,0,0,0,1,2.20


In [29]:
# X,y training
X_train = train_data.drop(columns = ['fantasy_points_ppr'])
y_train = train_data['fantasy_points_ppr']

# X,y testing
X_test = test_data.drop(columns = ['fantasy_points_ppr'])
y_test = test_data['fantasy_points_ppr']

## Train the model

In [39]:
rf = RandomForestRegressor(
    max_depth=50,
    max_features=0.8,
    min_samples_leaf=6,
    min_samples_split=20,
    n_estimators=200,
    random_state= 11, #pls sign micah parsons
    n_jobs = -1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

test_data['predicted'] = y_pred

# Overall RMSE

In [40]:
rmse_overall = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Overall RMSE: {rmse_overall:.2f}")

Overall RMSE: 8.13


## RMSE By Position

In [35]:
# un-dummify
position_cols = ['position_QB', 'position_RB', 'position_WR', 'position_TE']
test_data['position'] = test_data[position_cols].idxmax(axis=1)
test_data['position'] = test_data['position'].str.replace('position_', '')


rmse_by_position = (
    test_data.groupby('position')
             .apply(lambda g: np.sqrt(mean_squared_error(g['fantasy_points_ppr'], g['predicted'])))
)
print(rmse_by_position)

position
QB    8.116521
RB    8.186920
TE    7.990364
WR    8.168951
dtype: float64


<positron-console-cell-35>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<positron-console-cell-35>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


## RMSE By Week

In [38]:
# If week is part of the index
test_data = test_data.reset_index()  # moves 'week' from index to a column

# Now you can do RMSE by week
rmse_by_week = (
    test_data.groupby('week')
             .apply(lambda g: np.sqrt(mean_squared_error(g['fantasy_points_ppr'], g['predicted'])))
)

print("\nRMSE by week:")
print(rmse_by_week)


RMSE by week:
week
1     8.589298
2     7.913098
3     7.704117
4     8.518658
5     6.975970
6     9.935716
7     8.256423
8     7.487796
9     8.400921
10    6.364696
11    6.789813
12    7.394441
13    8.673290
14    8.769075
15    8.254480
16    9.243436
17    8.080945
dtype: float64
